# 01. Data pipeline: from API responses to the coded analytic sample

**From Augmentation to Replacement: Competing Frames of AI's Labour Impact
across News, Public, and Insider Discourse, 2022–2026**
Meshaal Kirmani · COMM424DA · Supervisor: Dr Chico Camargo

---

This notebook walks the full construction of the corpus and explains why each
step exists. It corresponds to **Section 3.2 (Corpus construction)** of the
report.

The pipeline runs in eight stages:

| # | Stage | Module | Output |
|---|---|---|---|
| 1 | Retrieval from three APIs | `retrieve_*.py`, `harvest_reddit.py` | `data/raw/*.jsonl` |
| 2 | Merge duplicate identifiers | `filter_clean.py` | (in memory) |
| 3 | Relevance rule (AI term **AND** labour term) | `utils.is_relevant` | (in memory) |
| 4 | English-only | `filter_clean.is_english` | (in memory) |
| 5 | Exact-title deduplication | `filter_clean.py` | (in memory) |
| 6 | Hacker News ≥5-comment threshold | `filter_clean.py` | `data/clean/*_clean.csv` |
| 7 | Balanced sampling (seed 42) | `build_annotation_sheet.py` | `data/clean/frozen_20260802/` |
| 8 | Three-axis coding + validation gate | `code_frames.py`, `validate_coding.py` | `data/coding/*.jsonl` |

**What can and cannot be re-run here.** Stages 2 to 8 re-run from the data in this
folder. Stage 1 cannot: it needs API keys, and the platforms' terms do not permit
redistributing raw responses, so `data/raw/` is documented rather than shipped in
full. Every count the retrieval stage produced is preserved in
`data/meta/funnel.json` and recomputed below where the inputs are present.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd, numpy as np, yaml
pd.set_option("display.width", 120)

cfg = yaml.safe_load(open(ROOT / "config.yaml"))
print("project root :", ROOT)
print("window       :", cfg["window"]["start"], "->", cfg["window"]["end"])

project root : /Users/meshaalkirmani/Documents/claude MSc ouptputs/final code submission
window       : 2022-11-01 -> 2026-05-31


## 1. Configuration is the single source of truth

Every parameter that shapes the corpus lives in `config.yaml`: the date window,
the search queries, the subreddit list, the relevance vocabulary, and the
thresholds. Nothing is hard-coded in the retrieval or cleaning modules, so the
corpus definition can be audited in one file.

In [2]:
print("query groups (68 queries in 6 groups):")
for g, qs in cfg["query_groups"].items():
    print(f"  {g:14s} {len(qs):2d}  e.g. {qs[:3]}")
print(f"\nevent-anchored pulls : {len(cfg['events'])}")
print(f"reddit backend       : {cfg['reddit']['backend']}")
print(f"HN comment threshold : {cfg['hackernews']['min_comments']}")

subs = sorted({s for tier in cfg["reddit"]["tiers"].values()
                 for cat in tier.values() for s in cat})
print(f"subreddits configured: {len(subs)}")

query groups (68 queries in 6 groups):
  core           10  e.g. ['AI jobs', 'AI employment', 'AI workforce']
  augmentation    8  e.g. ['AI copilot', 'AI augments', 'AI assistant productivity']
  replacement    12  e.g. ['AI replaces jobs', 'AI job losses', 'AI takes jobs']
  occupation     16  e.g. ['AI customer service', 'call center automation', 'AI radiologists']
  policy         10  e.g. ['universal basic income', 'AI regulation workers', 'AI unions']
  insider        12  e.g. ['AI coding assistants', 'AI replacing engineers', 'LLM hiring']

event-anchored pulls : 11
reddit backend       : arctic_shift
HN comment threshold : 5
subreddits configured: 49


## 2. The inclusion rule, which is what defines the corpus

An item is included **only if an AI term and a labour term co-occur in the same
text unit**, matched on word boundaries. This single regular expression is
evaluated locally, in project code, for all three sources.

**Why not use each platform's search?** The three APIs cannot be queried
equivalently. MediaCloud takes boolean queries over curated collections,
Algolia does ranked full-text retrieval, and the Reddit endpoint is a listing
service. Delegating relevance to them would mean comparing three retrieval
strategies rather than three ecosystems. One rule, one implementation, one place
to audit.

In [3]:
from utils import is_relevant

print("AI terms   :", cfg["relevance"]["ai_terms"][:110], "...")
print("labour terms:", cfg["relevance"]["labour_terms"][:110], "...")

AI terms   : \b(ai|a\.i\.|artificial intelligence|chatgpt|gpt-?[3-9][a-z]?|llms?|large language models?|generative ai|genai ...
labour terms: \b(jobs?|worker[s]?|workforce|workplace|labou?r|employ\w*|unemploy\w*|hir(e|es|ing)|lay-?offs?|laid off|career ...


### Why word boundaries are not a detail

An early version of the harvester matched query terms as plain substrings. The
two-letter token `ai` then matches inside **s-ai-d**, **em-ai-l**, **av-ai-lable**
and **ag-ai-n**, which mis-tagged roughly 79% of harvested posts with queries
they did not contain.

The cell below demonstrates both behaviours. The defect affected only the
*query tags* attached to items, never corpus membership, because the
word-boundary rule below, not the tags, is what decides inclusion. Rows were
re-tagged offline (`retag_harvest.py`); this is documented as deviation 7 in the
report.

In [4]:
import re
TRAP = ["He said the layoffs were coming",
        "Check your email about the job",
        "No roles available for graduates",
        "AI is replacing junior developers"]

sub_re  = re.compile(r"ai", re.I)                 # the defect
word_re = re.compile(r"\bai\b", re.I)             # the fix

print(f"{'text':46s} {'substring':>10s} {'word-bdry':>10s}")
for t in TRAP:
    print(f"{t:46s} {str(bool(sub_re.search(t))):>10s} {str(bool(word_re.search(t))):>10s}")

print("\nfull inclusion rule (AI term AND labour term) applied to the same lines:")
for t in TRAP:
    print(f"  {is_relevant(t, cfg)!s:5s}  {t}")

text                                            substring  word-bdry
He said the layoffs were coming                      True      False
Check your email about the job                       True      False
No roles available for graduates                     True      False
AI is replacing junior developers                    True       True

full inclusion rule (AI term AND labour term) applied to the same lines:
  False  He said the layoffs were coming
  False  Check your email about the job
  False  No roles available for graduates
  True   AI is replacing junior developers


## 3. The text unit

| Source | Unit coded |
|---|---|
| MediaCloud (news) | headline only |
| Reddit | title + first 100 words of body |
| Hacker News | title + first 100 words of body |

News is headline-only following Liu et al. (2019), who show frames are
recoverable from headlines, and because the headline is what most readers
actually encounter. HTML entities and tags are stripped so that the human coder
and the model receive byte-identical text.

The difference that remains is a real threat to comparability: news units have
a median of 13 words against 106 to 109 for the forums. It is addressed by the
length-split robustness check in notebook 02.

In [5]:
from build_annotation_sheet import clean_text

raw = "AI &amp; jobs: <p>firms cut 10% of staff</p>&nbsp;after rollout"
print("raw    :", raw)
print("cleaned:", clean_text(raw))

raw    : AI &amp; jobs: <p>firms cut 10% of staff</p>&nbsp;after rollout
cleaned: AI & jobs:
firms cut 10% of staff
 after rollout


## 4. Cleaning order, and the attrition it produces

The order matters and is fixed: deduplicating before filtering is not the same
operation as filtering before deduplicating, and early experiments with the
order shifted the final counts. The sequence in `filter_clean.process()` is:

```
merge duplicate ids  ->  normalise dates  ->  relevance rule
   ->  English only  ->  exact-title dedupe  ->  HN >= 5 comments  ->  flags
```

`src/compute_funnel.py` replays exactly that sequence over the raw files and
writes `data/meta/funnel.json`. The counts below are read from it, so the figure
and the report's Figure 1 cannot disagree with the code.

In [6]:
funnel = json.loads((ROOT / "data/meta/funnel.json").read_text())
S, PS = funnel["stages"], funnel["per_source"]

rows = [("API responses retrieved", S["retrieved"], "3 APIs, 68 queries, 11 event pulls"),
        ("Unique items",            S["unique"],    "same post found by several queries"),
        ("AI term AND labour term", S["relevant"],  "the inclusion rule"),
        ("English only",            S["english"],   "cross-source comparability"),
        ("Duplicate titles removed", S["deduped"],  "syndication and cross-posts"),
        ("Clean corpus",            S["clean"],     "HN >= 5 comments"),
        ("Coded sample",            S["coded"],     "600/ecosystem, half-year balanced"),
        ("Analytic sample",         S["analysis"],  "tangential excluded")]

prev = None
print(f"{'stage':26s} {'n':>9s} {'kept':>6s}   why")
for name, n, why in rows:
    kept = "" if prev is None else f"{100*n/prev:5.1f}%"
    print(f"{name:26s} {n:>9,} {kept:>6s}   {why}")
    prev = n

print("\nretrieval precision by source (% of retrieved reaching the clean corpus):")
for s, v in PS.items():
    print(f"  {s:12s} {v['retrieved']:>8,} -> {v['clean']:>7,}   {v['survival_pct']:5.1f}%")

stage                              n   kept   why
API responses retrieved      328,270          3 APIs, 68 queries, 11 event pulls
Unique items                 292,845  89.2%   same post found by several queries
AI term AND labour term       48,646  16.6%   the inclusion rule
English only                  46,945  96.5%   cross-source comparability
Duplicate titles removed      44,047  93.8%   syndication and cross-posts
Clean corpus                  40,735  92.5%   HN >= 5 comments
Coded sample                   1,800   4.4%   600/ecosystem, half-year balanced
Analytic sample                1,205  66.9%   tangential excluded

retrieval precision by source (% of retrieved reaching the clean corpus):
  mediacloud    201,900 ->  12,230     6.1%
  hackernews      9,393 ->     951    10.1%
  reddit        116,977 ->  27,554    23.6%


**Reading the funnel.** The inclusion rule is where the corpus is actually
defined: 83% of unique items are discarded there. Retrieval precision then
differs almost fourfold by source (Reddit 23.6%, Hacker News 10.1%, news 6.1%),
because MediaCloud matches whole articles at collection level while the Reddit
harvest is already community-scoped.

Hacker News is small in absolute terms because the five-comment threshold
removes 78% of otherwise-eligible stories (4,263 → 951). That threshold selects
*discursive* rather than purely informational posts, and it is applied to Hacker
News only. I report this difference between sources in the paper instead of
smoothing it over.

In [7]:
clean = {}
for s in ("mediacloud", "reddit", "hackernews"):
    clean[s] = pd.read_csv(ROOT / f"data/clean/{s}_clean.csv", low_memory=False)
    print(f"{s:12s} {len(clean[s]):>7,} items")
total = sum(len(v) for v in clean.values())
print(f"{'TOTAL':12s} {total:>7,} items")
assert total == 40_735, f"expected 40,735, got {total:,}"
print("\nmatches the 40,735 reported in the paper")

mediacloud    12,230 items


reddit        27,554 items
hackernews       951 items
TOTAL         40,735 items

matches the 40,735 reported in the paper


## 5. Provenance flags, and how they make selection bias measurable

Frame-loaded queries ("AI layoffs") could manufacture the very distribution RQ1
reports. Rather than assume this away, every item records which query groups
surfaced it, and two flags are derived:

- `in_neutral_baseline`: matched at least one core (frame-neutral) query.
  Restricting to these items gives a subset whose retrieval carries no frame
  loading, and RQ3 is re-run on it as a robustness check.
- `keyword_matched`: matched any keyword query at all. Listing-harvested
  Reddit items can pass the relevance rule while matching no query, giving a
  keyword-independent subsample.

This matters because the neutral-baseline share differs sharply by ecosystem, so
a naive comparison would partly compare retrieval strategies.

In [8]:
for s, d in clean.items():
    print(f"{s:12s} neutral-baseline {d.in_neutral_baseline.mean():.2f}   "
          f"keyword-matched {d.keyword_matched.mean():.2f}")

mediacloud   neutral-baseline 0.98   keyword-matched 1.00
reddit       neutral-baseline 0.60   keyword-matched 0.94
hackernews   neutral-baseline 0.53   keyword-matched 1.00


## 6. Sampling: collect a large corpus, then sample from it

The corpus is roughly 23 times the coded sample, so the sample can be redrawn
without re-collecting. Six hundred items per ecosystem were drawn, balanced
across half-years, under seed 42.

I stratified on ecosystem, half-year and source outlet only, and never on query
group or anything else related to the frame, so that frame metadata cannot
influence which items get coded. Caps of 8 items per subreddit and 6 per outlet
are applied *globally* to stop any single community dominating.

Because the sample is balanced rather than proportional it is **not
self-weighting**: population weights range 0.12–1.80. Whole-corpus proportions
would need weighting; within-source and within-period proportions do not, and
only the latter are reported.

In [9]:
frozen = ROOT / "data/clean/frozen_20260802"
samp = pd.concat([pd.read_csv(frozen / f"{s}_sample.csv").assign(source=s)
                  for s in ("mediacloud", "reddit", "hackernews")], ignore_index=True)
print("coded sample:", len(samp), "items\n")
print(pd.crosstab(samp.source, samp.half_year).to_string())
print("\nper source:", samp.source.value_counts().to_dict())

coded sample: 1800 items

half_year   2022H2  2023H1  2023H2  2024H1  2024H2  2025H1  2025H2  2026H1
source                                                                    
hackernews      13      79      65      75      75      92     101     100
mediacloud      75      75      75      75      75      75      75      75
reddit          75      75      75      75      75      75      75      75

per source: {'mediacloud': 600, 'reddit': 600, 'hackernews': 600}


## 7. Coding on three axes

**Axis 1 (primary).** `claude-opus-5` codes every item against the codebook,
which is supplied as a cached prompt prefix. Structured outputs constrain each
field to its codebook value set *at decode time*, so an out-of-vocabulary label
is impossible rather than merely unlikely. Refusals are detected before any
content is read and are never written as codes. A refusal quietly turned into a
label would corrupt the reliability estimate.

**Axis 2 (cross-check).** GABRIEL-ratings on `gpt-4o`, coarse frame only. This
differs from the primary coder on **both** model family and measurement
formulation, so agreement between them is stronger evidence than agreement
between two prompts of one model.

**Axis 3 (unsupervised).** BERTopic over a stratified subsample of 6,951
documents, as a structure derived without reference to the codebook.

Total API spend across all axes: about $19.

In [10]:
from code_frames import VALID, MODEL
print("primary coder:", MODEL, "\n")
for k, v in VALID.items():
    print(f"  {k:14s} {v}")

primary coder: claude-opus-5 

  coarse_frame   ['REPLACEMENT', 'AUGMENTATION', 'TRANSFORMATION', 'MIXED', 'TANGENTIAL']
  problem        ['JOB_LOSS', 'SKILL_CHANGE', 'OPPORTUNITY', 'INEQUALITY_POWER', 'NONE']
  cause          ['TECHNOLOGY', 'FIRMS', 'POLICY', 'WORKERS', 'NONE']
  moral          ['NEGATIVE', 'POSITIVE', 'AMBIVALENT', 'NEUTRAL', 'NONE']
  treatment      ['REGULATE', 'ADAPT', 'REDISTRIBUTE', 'ACCELERATE', 'NONE']


In [11]:
calls = [json.loads(l) for l in open(ROOT / "data/coding/llm_calls.jsonl")]
print(f"logged calls: {len(calls):,}")
ex = {k: v for k, v in calls[0].items() if k in
      ("id", "model", "stop_reason", "system_sha", "usage")}
print("\nexample audit record:")
print(json.dumps(ex, indent=2)[:520])

logged calls: 1,800

example audit record:
{
  "id": "mc_8761fdef3681b4f5c4143b319d6e58fb5537ba7182efc7218a49d767f1c64b74",
  "model": "claude-opus-5",
  "stop_reason": "end_turn",
  "system_sha": "e1e89b41c7cfdb0f",
  "usage": {
    "input": 39,
    "output": 164,
    "cache_read": 0,
    "cache_write": 3523
  }
}


## 8. The validation gate

The proposal fixed the criterion **before** any coding: Cohen's κ ≥ 0.60 on at
least three of the four Entman functions, failing which the analysis would fall
back to a coarse three-category typology at κ ≥ 0.55 after one documented
codebook-refinement cycle.

I hand-coded 300 items (100 per ecosystem) with the **source label hidden**.
Blinding mattered because I expected Reddit to be more replacement-framed, and
that expectation would otherwise have been easy to confirm by accident.

In [12]:
from analysis import load_human, CODE_COLS
from sklearn.metrics import cohen_kappa_score

m = load_human()
print(f"validation items: {len(m)}\n")
print(f"{'variable':14s} {'kappa':>7s} {'agreement':>10s} {'gap':>7s}")
res = {}
for c in CODE_COLS:
    a, b = m[f"{c}_h"], m[f"{c}_m"]
    k = cohen_kappa_score(a, b, labels=sorted(set(a) | set(b)))
    agr = (a == b).mean()
    res[c] = k
    print(f"{c:14s} {k:7.3f} {agr:10.3f} {agr-k:7.3f}")

passing = [c for c in ["problem", "cause", "moral", "treatment"] if res[c] >= 0.60]
print(f"\nfunctions at/above 0.60: {len(passing)}/4  {passing}")
print("DECISION:", "PASS, full four-function schema retained"
      if len(passing) >= 3 else "FAIL, fallback required")

validation items: 300

variable         kappa  agreement     gap
coarse_frame     0.797      0.860   0.063
problem          0.834      0.887   0.053
cause            0.792      0.857   0.065
moral            0.803      0.853   0.051
treatment        0.591      0.877   0.286

functions at/above 0.60: 3/4  ['problem', 'cause', 'moral']
DECISION: PASS, full four-function schema retained


**On treatment recommendation.** Its κ of 0.591 is the one value below the
threshold, and the reason is prevalence rather than coding quality. Raw
agreement on treatment is 0.877, nearly the highest of the five variables, but
roughly four-fifths of items recommend nothing, so agreement expected by chance
is high and the correction severe. The diagnostic is the **gap** column above:
0.286 for treatment against 0.050–0.065 for everything else. This is the kappa
paradox, and both numbers are reported so the coefficient is not misread as
unreliable coding.

## 9. What feeds the analysis

| File | Contents |
|---|---|
| `data/clean/*_clean.csv` | clean corpus, 40,735 items |
| `data/clean/frozen_20260802/` | the frozen coded sample, 1,800 items |
| `data/coding/llm_codes_full1800.jsonl` | primary model codes |
| `data/coding/llm_codes_gold300.jsonl` | model codes for the validation subset |
| `data/annotation/annotation_sheet_filledin.xlsx` | my 300 hand codes |
| `data/coding/gabriel_codes_gold300.jsonl` | cross-check model codes |

`frozen_20260802` is a dated snapshot. Every number in the submitted report
refers to that copy and no other, so the analysis describes a fixed object
rather than whatever the pipeline last produced.

**Continue in `02_analysis_and_results.ipynb`**, which reproduces every table,
figure and quoted number in the report from these files.